In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

In [2]:
symbol = 'MSFT'
data = yf.download(symbol, start='2022-04-01', end='2023-05-01')
data.drop(['Open', 'High', 'Low', 'Volume','Close'], inplace=True, axis=1)

initial_cash = 10_000_000
commission = 0.02

print(data.head())

[*********************100%%**********************]  1 of 1 completed

             Adj Close
Date                  
2022-04-01  303.372375
2022-04-04  308.813934
2022-04-05  304.803864
2022-04-06  293.646271
2022-04-07  295.479736


In [3]:
def CalcRSI(data, window=2):
    delta = data['Adj Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean(2)
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean(2)
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

data['RSI'] = CalcRSI(data)

In [4]:
data['Position'] = np.where(data['RSI'] < 10, 1, 0)
data['Position'] = np.where(data['RSI'] > 90, -1, data['Position'])

In [5]:
data['Daily Return'] = data['Adj Close'].pct_change()
data['Strategy Return'] = data['Daily Return'] * data['Position'].shift(1)

In [6]:
data['Cumulative Market Return'] = (1 + data['Daily Return']).cumprod()
data['Cumulative Strategy Return'] = (1 + data['Strategy Return']).cumprod()

In [7]:
data['Cash'] = initial_cash
data['Position Value'] = data['Position'].shift(1) * data['Adj Close']
data['Total Value'] = data['Cash'] + data['Position Value']
data['Cash'] = data['Total Value'] - (data['Position'].diff() * data['Adj Close']).fillna(0) * (1 + commission)

In [8]:
final_cash_rsi = data['Total Value'].iloc[-1]
profit_loss_rsi = final_cash_rsi - initial_cash

In [9]:
data['Rolling Max'] = data['Total Value'].cummax()
data['Drawdown'] = (data['Total Value'] - data['Rolling Max']) / data['Rolling Max']
max_drawdown_rsi = data['Drawdown'].min()


In [10]:
data['Buy and Hold Value'] = initial_cash * (1 + data['Daily Return']).cumprod()
final_cash_bh = data['Buy and Hold Value'].iloc[-1]
profit_loss_bh = final_cash_bh - initial_cash

In [11]:
data['Rolling Max BH'] = data['Buy and Hold Value'].cummax()
data['Drawdown BH'] = (data['Buy and Hold Value'] - data['Rolling Max BH']) / data['Rolling Max BH']
max_drawdown_bh = data['Drawdown BH'].min()

In [12]:
print(f"RSI(2) Strategy - Final Cash Balance: ${final_cash_rsi:,.2f}")
print(f"RSI(2) Strategy - Profit/Loss: ${profit_loss_rsi:,.2f}")
print(f"RSI(2) Strategy - Max Drawdown: {max_drawdown_rsi * 100:.2f}%\n")

print(f"Buy and Hold Strategy - Final Cash Balance: ${final_cash_bh:,.2f}")
print(f"Buy and Hold Strategy - Profit/Loss: ${profit_loss_bh:,.2f}")
print(f"Buy and Hold Strategy - Max Drawdown: {max_drawdown_bh * 100:.2f}%\n")

RSI(2) Strategy - Final Cash Balance: $9,999,695.79
RSI(2) Strategy - Profit/Loss: $-304.21
RSI(2) Strategy - Max Drawdown: -0.01%

Buy and Hold Strategy - Final Cash Balance: $10,027,613.18
Buy and Hold Strategy - Profit/Loss: $27,613.18
Buy and Hold Strategy - Max Drawdown: -31.67%

